# 07 · Looking inside (Block G)

Everything the previous notebooks did left state somewhere. EDC runtimes are stateless processes; their state lives in two places per participant:

| Store | Holds |
|---|---|
| **Vault** | secrets: private keys, AES keys, STS client secrets, backend credentials |
| **Postgres** (one database per participant) | everything else: assets, policies, negotiations, agreements, transfers, EDRs, credentials, DID documents… |

In [1]:
import dataspace as ds

## G1 · Vault

**Concept:** HashiCorp Vault stores secrets behind an HTTP API. Here each participant has its own Vault with a **KV v2** engine mounted at `secret/`.
EDC never stores a secret in its database; it stores an **alias** and resolves it in Vault when needed.
This local Vault uses the root token `root` and **in-memory storage**: a Vault pod restart loses every key.

**What:** `GET /<p>/vault/v1/secret/metadata?list=true`. KV v2 separates `metadata/` (names, versions) from `data/` (values).  **Returns:** the secret names.

In [2]:
for participant in ds.PARTICIPANTS:
    listing = ds.get(f"{ds.BASE_URL}/{participant}/vault/v1/secret/metadata",
                     params={"list": "true"}, headers={"X-Vault-Token": ds.VAULT_TOKEN}).json()
    keys = listing["data"]["keys"]
    print(f"   {participant}: {len(keys)} secrets, e.g. {keys[:5]}\n")

GET    /authority/vault/v1/secret/metadata  ->  200
   authority: 6 secrets, e.g. ['authority', 'authority-aes', 'authority-pub', 'did%3Aweb%3Aauthority-identityhub%253A8383%3Aapi%3Adid-apikey', 'did%3Aweb%3Aauthority-identityhub%253A8383%3Aapi%3Adid-sts-client-secret']

GET    /provider/vault/v1/secret/metadata  ->  200
   provider: 25 secrets, e.g. ['devstoreaccount1', 'devstoreaccount1-key1', 'did%3Aweb%3Aprovider-identityhub%253A8383%3Aapi%3Adid-apikey', 'did%3Aweb%3Aprovider-identityhub%253A8383%3Aapi%3Adid-sts-client-secret', 'oauth-secret']

GET    /consumer/vault/v1/secret/metadata  ->  200
   consumer: 24 secrets, e.g. ['consumer', 'consumer-aes', 'consumer-pub', 'devstoreaccount1', 'devstoreaccount1-key1']



What the names mean:

| Secret | Is |
|---|---|
| `<p>` / `<p>-pub` | the participant's EC private / public key (PEM). Signs tokens, credentials (authority) and EDRs |
| `<p>-aes` | AES key EDC uses to encrypt participant-context configuration |
| `<did>-sts-client-secret` | secret the control plane uses to get tokens from its wallet's STS |
| `<did>-apikey` | API key of the participant context in the identity hub |
| `transfer-process-<id>-data-address` (provider) | per-transfer data addresses kept by the data plane |
| `event-hub-connection-string` (authority) | used by the telemetry service to mint SAS tokens (written by the E2E suite) |

**What:** `GET /<p>/vault/v1/secret/data/<name>`. The public key is safe to print:

In [3]:
secret = ds.get(f"{ds.BASE_URL}/provider/vault/v1/secret/data/provider-pub", headers={"X-Vault-Token": ds.VAULT_TOKEN}).json()
print(secret["data"]["data"]["content"])
print("version:", secret["data"]["metadata"]["version"])

GET    /provider/vault/v1/secret/data/provider-pub  ->  200
-----BEGIN PUBLIC KEY-----
MFkwEwYHKoZIzj0CAQYIKoZIzj0DAQcDQgAEJ2TM59OS4Uy5GpmCtKL0gftCsyVA
qXEPi/CaNivhq1V/1L8y3sijRQTro8n8fEJBIkmRB9xX2SEtN6v0akHpZg==
-----END PUBLIC KEY-----
version: 1


## G2 · Postgres

**Concept:** one Postgres server, one database per tenant (`authoritydb`, `providerdb`, `consumerdb`, `billingdb`). All runtimes of a participant share its database.
EDC creates its tables at startup. Postgres has no volume in this setup, so recreating its pod wipes everything.

In [4]:
print("databases:", ds.psql("postgres", "select string_agg(datname, ', ') from pg_database where not datistemplate"))
print()
print("consumerdb tables:")
print(ds.psql("consumerdb", "select string_agg(table_name, ', ' order by table_name) from information_schema.tables where table_schema='public'"))

databases: postgres, authoritydb, consumerdb, billingdb, providerdb

consumerdb tables:
credential_resource, did_resources, edc_agreement_retirement, edc_asset, edc_contract_agreement, edc_contract_definitions, edc_contract_negotiation, edc_credential_offers, edc_data_plane_instance, edc_edr_entry, edc_holder_credentialrequest, edc_jti_validation, edc_lease, edc_policy_monitor, edc_policydefinitions, edc_sts_client, edc_telemetry_record, edc_transfer_process, keypair_resource, participant_context


Mapping tables to what you did:

| Table | Written by | Notebook |
|---|---|---|
| `participant_context`, `keypair_resource`, `did_resources` | identity hub | 00 (DID documents) |
| `credential_resource`, `edc_holder_credentialrequest` | identity hub (wallet) | 01 |
| `edc_asset`, `edc_policydefinitions`, `edc_contract_definitions` | control plane | 02 (on the provider) |
| `edc_contract_negotiation`, `edc_contract_agreement` | control plane | 04 |
| `edc_transfer_process`, `edc_edr_entry`, `edc_policy_monitor` | control plane | 05 |
| `edc_agreement_retirement` | DSE retirement extension | 05 |
| `edc_telemetry_record` | data plane / telemetry agent | 05–06 |
| `edc_lease` | every store: row locks so that only one runtime instance processes an entity at a time | — |

In [5]:
for table in ["edc_asset", "edc_contract_negotiation", "edc_contract_agreement", "edc_transfer_process", "edc_edr_entry", "edc_telemetry_record"]:
    provider_count = ds.psql("providerdb", f"select count(*) from {table}")
    consumer_count = ds.psql("consumerdb", f"select count(*) from {table}")
    print(f"   {table:26} provider: {provider_count:>5}   consumer: {consumer_count:>5}")

   edc_asset                  provider:    48   consumer:     0


   edc_contract_negotiation   provider:    27   consumer:    30


   edc_contract_agreement     provider:    27   consumer:    27


   edc_transfer_process       provider:    35   consumer:    35


   edc_edr_entry              provider:     0   consumer:    17


   edc_telemetry_record       provider:     0   consumer:   170


### State machines are stored as numbers

The Management API returns state names; the database stores codes. The ones you met in this series:

In [6]:
NEGOTIATION_STATES = {1200: "FINALIZED", 1400: "TERMINATED"}
TRANSFER_STATES = {600: "STARTED", 800: "COMPLETED", 850: "TERMINATED"}

print("consumer negotiations:")
for row in ds.psql("consumerdb", "select state, count(*) from edc_contract_negotiation group by state order by state").splitlines():
    code, count = row.split("|")
    print(f"   {code} {NEGOTIATION_STATES.get(int(code), '?'):11} {count}")

print("consumer transfers:")
for row in ds.psql("consumerdb", "select state, count(*) from edc_transfer_process group by state order by state").splitlines():
    code, count = row.split("|")
    print(f"   {code} {TRANSFER_STATES.get(int(code), '?'):11} {count}")

consumer negotiations:


   1200 FINALIZED   27
   1400 TERMINATED  3
consumer transfers:
   600 STARTED     17
   800 COMPLETED   1
   850 TERMINATED  17


The billing database, with the participant names used by the JWT roles in notebook 06:

In [7]:
print(ds.psql("billingdb", "select string_agg(name || ' <' || email || '>', ', ') from participant_id"))

provider <provider@example.com>, consumer <consumer@example.com>


## G3 · Logs

Each runtime logs at DEBUG level. Following a flow in the logs is often the fastest way to see *why* something happened.
These are the provider control plane's last negotiation outcomes:

In [8]:
log_lines = ds.kubectl("logs", "deploy/provider-controlplane", "--since=15m").splitlines()
outcomes = [line for line in log_lines if "ContractNegotiation" in line and ("FINALIZED" in line or "TERMINATED" in line)]
for line in outcomes[-6:]:
    print(line)

[2026-10-09 14:04:00] [FINE   ] [ProviderContractNegotiationManagerImpl] ContractNegotiation 784d3602-a4ef-459c-b101-4b67afa5bcce is now in state FINALIZED
[2026-10-09 14:04:19] [FINE   ] [ProviderContractNegotiationManagerImpl] ContractNegotiation 5bc17d89-fac9-455f-ab83-cfd81fe7bafd is now in state FINALIZED
[2026-10-09 14:06:28] [FINE   ] [ProviderContractNegotiationManagerImpl] ContractNegotiation f3263c0e-a946-4fd7-95c2-838dc94b3f33 is now in state FINALIZED
[2026-10-09 14:06:53] [FINE   ] [ProviderContractNegotiationManagerImpl] ContractNegotiation aa6b7aca-5026-4d53-bbb2-3283211282d3 is now in state FINALIZED


Other useful ones: `deploy/consumer-controlplane` (the consumer's side of negotiations and transfers), `deploy/provider-dataplane` (data requests served),
`deploy/authority-issuerservice` (credential issuance), `deploy/authority-federatedcatalog` (crawls).

## Everything this run created

In [9]:
print(ds.STATE_FILE.read_text())

{
  "run_id": "1d0d4e",
  "data_asset": "ging-data-1d0d4e",
  "nobody_asset": "ging-nobody-1d0d4e",
  "travel_asset": "ging-travel-1d0d4e",
  "short_lived_asset": "ging-20s-1d0d4e",
  "consumer_agreement_id": "599d9a9f-a70c-43e1-9a6a-e34d115cf6a1",
  "provider_agreement_id": "2228df2c-0d34-4061-b07f-7d7f5b02173f",
  "transfer_ids": [
    "f5a76299-6c79-4bcf-b248-e840f3412a99",
    "73356fdd-7f15-42ec-a799-84f33551a93e",
    "61bcbd2c-457a-4166-bc80-b51c4ba52950"
  ]
}
